## Instrucciones de ejecución

1. Ejecutar la celda de instalación a continuación.
2. Colocar `chb20_12.edf` en `archivos/` (un nivel arriba de este notebook).
3. Ejecutar todas las celdas en orden (`Kernel → Restart & Run All`).

**Librerías necesarias:**
```
pip install numpy scipy matplotlib pyedflib ipykernel
```

In [ ]:
%pip install numpy scipy matplotlib pyedflib ipykernel --quiet

# Práctica 2 — Análisis Espectral de Crisis Epilépticas

**Algorítmica y Lógica Computacional — UCA**  
Berkelaar · Caitano · Flachsland

---

## Objetivo

Aplicar técnicas de análisis espectral a señales EEG del dataset **CHB-MIT Scalp EEG Database** (PhysioNet) para detectar y caracterizar crisis epilépticas. Se trabaja con el registro `chb20_12.edf` del paciente 20, cuya crisis fue anotada entre los segundos 94 y 123.

---

## Dataset y segmentación

El EEG se segmenta en tres bloques según la anotación de la crisis:

| Bloque | Duración  | Descripción                        |
|--------|-----------|---------------------------------|
| Before | 2 minutos | Actividad basal previa a la crisis |
| Crisis | 29 s      | Segmento ictal anotado             |
| After  | 2 minutos | Recuperación post-crisis           |

A cada segmento se le resta la media por canal para eliminar offset DC.

## Bandas cerebrales analizadas

| Banda  | Rango (Hz) | Asociación funcional                          |
|--------|-----------|-----------------------------------------------|
| Delta  | 0–4        | Sueño profundo, ondas lentas patológicas      |
| Theta  | 4–8        | Somnolencia, memoria, actividad hipocampal    |
| Alpha  | 8–12       | Relajación con ojos cerrados, ritmo de Berger |
| Beta   | 12–30      | Alerta, cognición activa, actividad motora    |
| Gamma  | 30–64      | Procesamiento sensorial de alta frecuencia    |

## Métodos espectrales

- **FFT**: estimación directa del espectro de magnitud. Resolución $\Delta f = f_s/N$. Sin ventaneo → susceptible a *leakage*.
- **PSD de Welch**: promedia periodogramas de segmentos solapados con ventana Hann. Reduce varianza a costa de resolución.
- **Periodograma**: estimador directo $P(f) = |X(f)|^2 / (N f_s)$. Alta varianza, revela detalles finos.
- **STFT / Espectrograma**: aplica FFT a ventanas deslizantes → mapa tiempo-frecuencia.

## Estrategia de visualización multi-canal

Con 28 canales simultáneos, superponer todas las curvas genera gráficos ilegibles. Se adoptó **media ± 1σ**: la línea gruesa es la media entre los 28 canales y la banda sombreada indica ±1 desviación estándar. Esto preserva la información de todos los canales sin saturar la visualización.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy import signal
from pyedflib import highlevel

# ── Configuración ──────────────────────────────────────────────────────────────
BASE_DIR   = Path(".").resolve()
EDF_PATH   = BASE_DIR.parent / "archivos" / "chb20_12.edf"
if not EDF_PATH.exists():
    EDF_PATH = BASE_DIR / "archivos" / "chb20_12.edf"

START_SEC  = 94    # inicio crisis anotada (s)
END_SEC    = 123   # fin crisis anotada (s)
WINDOW_SEC = 120   # duración before/after (2 min)

BANDS = {
    "delta": (0,  4),
    "theta": (4,  8),
    "alpha": (8,  12),
    "beta":  (12, 30),
    "gamma": (30, 64),
}
COLORS      = {"before": "steelblue", "crisis": "tomato", "after": "seagreen"}
BAND_COLORS = {
    "delta": "#4c78a8", "theta": "#f58518",
    "alpha": "#54a24b", "beta":  "#e45756", "gamma": "#8e6bbf",
}
BLOQUES = ["before", "crisis", "after"]

# ── Carga EDF ──────────────────────────────────────────────────────────────────
def cargar_edf(path):
    sigs, sig_hdrs, _ = highlevel.read_edf(str(path))
    fs     = int(sig_hdrs[0].get("sample_frequency", sig_hdrs[0].get("sample_rate")))
    labels = [h.get("label", f"Canal {i}") for i, h in enumerate(sig_hdrs)]
    return sigs, fs, labels

def extract_center(sig, start, end):
    seg = sig[:, start:end]
    return seg - seg.mean(axis=1, keepdims=True)

def preparar_segmentos(signals, fs):
    s0  = START_SEC * fs;  s1  = END_SEC * fs
    win = WINDOW_SEC * fs; N   = signals.shape[1]
    i0  = max(0, s0 - win); i1 = min(N, s1 + win)
    before = extract_center(signals, i0, s0)
    crisis = extract_center(signals, s0, s1)
    after  = extract_center(signals, s1, i1)
    total  = np.concatenate((before, crisis, after), axis=1)
    refs   = {
        "crisis_ini_total_sec": before.shape[1] / fs,
        "crisis_fin_total_sec": (before.shape[1] + crisis.shape[1]) / fs,
    }
    return {"before": before, "crisis": crisis, "after": after, "total": total}, refs

# ── Funciones espectrales ──────────────────────────────────────────────────────
def calcular_fft(seg, fs):
    n = seg.shape[1]
    return np.fft.rfftfreq(n, d=1/fs), np.abs(np.fft.rfft(seg, axis=1)) / n

def calcular_psd_welch(seg, fs, nperseg=None, overlap=0.5):
    if nperseg is None: nperseg = min(4 * fs, seg.shape[1])
    return signal.welch(seg, fs=fs, axis=1, nperseg=nperseg,
                        noverlap=int(nperseg * overlap), scaling="density")

def calcular_periodograma(seg, fs):
    return signal.periodogram(seg, fs=fs, axis=1, scaling="density")

def calcular_stft(seg, fs, window_sec=2, overlap=0.50):
    nperseg = min(int(window_sec * fs), seg.shape[1])
    freqs, times, zxx = signal.stft(seg, fs=fs, axis=1, nperseg=nperseg,
                                    noverlap=int(nperseg * overlap), boundary=None)
    return freqs, times, np.abs(zxx) ** 2

def calcular_espectrograma(seg, fs, window_sec, overlap):
    nperseg = min(int(window_sec * fs), seg.shape[1])
    return signal.spectrogram(seg, fs=fs, axis=1, nperseg=nperseg,
                              noverlap=int(nperseg * overlap),
                              scaling="density", mode="psd")

def potencia_por_banda(freqs, psd):
    result = {}
    for name, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs < hi)
        result[name] = (np.trapezoid(psd[:, mask], freqs[mask], axis=1)
                        if np.any(mask) else np.zeros(psd.shape[0]))
    return result

def potencia_bandas_tiempo(freqs, spec):
    result = {}
    for name, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs < hi)
        result[name] = (np.trapezoid(spec[:, mask, :], freqs[mask], axis=1)
                        if np.any(mask) else np.zeros((spec.shape[0], spec.shape[2])))
    return result

def frecuencias_dominantes(freqs, spectrum, fmax=64, top_n=5):
    mask = (freqs > 0) & (freqs <= fmax)
    mean_spec = spectrum[:, mask].mean(axis=0)
    idx = np.argsort(mean_spec)[-top_n:][::-1]
    return list(zip(freqs[mask][idx], mean_spec[idx]))

# ── Cargar datos ───────────────────────────────────────────────────────────────
print(f"Cargando: {EDF_PATH}")
signals, fs, labels = cargar_edf(EDF_PATH)
segmentos, refs = preparar_segmentos(signals, fs)

print(f"Frecuencia de muestreo : {fs} Hz")
print(f"Canales                : {signals.shape[0]}")
print(f"Crisis anotada         : {START_SEC}s – {END_SEC}s  ({END_SEC-START_SEC}s de duración)")
for nombre, seg in segmentos.items():
    print(f"  {nombre:<7}: forma {seg.shape}  ({seg.shape[1]/fs:.1f} s)")
print(f"Crisis en bloque total : {refs['crisis_ini_total_sec']:.1f}s – {refs['crisis_fin_total_sec']:.1f}s")

---
# Escenario 1 — Análisis por Bloques

Se analiza cada bloque (Before / Crisis / After) de forma independiente, tratándolo como una señal cuasi-estacionaria.

## Fundamento teórico

### FFT vs PSD de Welch

La **FFT** calcula el espectro de magnitud directamente sobre todo el segmento con resolución $\Delta f = f_s/N$. No aplica ventaneo, por lo que es susceptible a *spectral leakage*.

La **PSD de Welch** divide la señal en $K$ segmentos solapados, aplica ventana Hann y promedia los periodogramas:
$$P_{\text{Welch}}(f) = \frac{1}{K}\sum_{k=1}^{K} |X_k(f)|^2$$
Reduce varianza espectral a costa de resolución. Ideal para señales no estacionarias como el EEG.

### Periodograma
Estimador directo: $P(f) = |X(f)|^2 / (N f_s)$. Alta varianza, pero revela detalles espectrales finos.

### STFT y espectrograma
$$\text{STFT}(m, k) = \sum_n x[n]\,w[n-mH]\,e^{-j2\pi kn/N}$$
El espectrograma $|\text{STFT}|^2$ es un mapa tiempo-frecuencia. El tamaño de ventana $W$ y el solapamiento $H$ definen el balance resolución temporal-frecuencial:
- **Ventana grande**: mejor resolución frecuencial, peor temporal.
- **Overlap alto**: más suavizado temporal, mayor costo computacional.

## Complejidad algorítmica — Escenario 1

| Método       | Complejidad             | $C$ = canales | Total por escenario           |
|--------------|-------------------------|---------------|-------------------------------|
| FFT          | $O(N \log N)$           | 28            | $O(C \cdot N \log N)$         |
| PSD Welch    | $O(K \cdot M \log M)$   | 28            | $O(C \cdot K \cdot M \log M)$ |
| Periodograma | $O(N \log N)$           | 28            | $O(C \cdot N \log N)$         |
| STFT         | $O(T \cdot M \log M)$   | 28            | $O(C \cdot T \cdot M \log M)$ |
| Potencia/banda | $O(F \cdot B)$        | 28            | $O(C \cdot F \cdot B)$        |

**¿Mejora la función `random`?** El zero-padding a potencia de 2 (`np.fft.rfft(x, n=next_pow2)`) no cambia el orden asintótico $O(N\log N)$, pero puede reducir la constante oculta cuando $N$ es un número primo o con factores grandes. NumPy aplica internamente estas optimizaciones (FFTPACK), por lo que el beneficio en la práctica es marginal.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E1 — FFT y PSD por bloques + detección de frecuencias no deseadas
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=False)
fig.suptitle(
    "Escenario 1 — FFT y PSD por bloques\n"
    "(línea gruesa: media entre 28 canales | banda: ±1 desv. estándar)",
    fontsize=13, fontweight="bold",
)

for name in BLOQUES:
    seg, color = segmentos[name], COLORS[name]
    f_fft, fft_mag = calcular_fft(seg, fs)
    f_psd, psd     = calcular_psd_welch(seg, fs)

    mask_f = f_fft <= 64
    mu_f   = fft_mag[:, mask_f].mean(axis=0)
    sd_f   = fft_mag[:, mask_f].std(axis=0)
    axes[0].fill_between(f_fft[mask_f], mu_f - sd_f, mu_f + sd_f, color=color, alpha=0.25)
    axes[0].plot(f_fft[mask_f], mu_f, color=color, linewidth=2.0, label=name)

    mask_p = f_psd <= 64
    mu_p   = psd[:, mask_p].mean(axis=0)
    sd_p   = psd[:, mask_p].std(axis=0)
    axes[1].fill_between(f_psd[mask_p], np.maximum(mu_p - sd_p, 1e-12), mu_p + sd_p,
                         color=color, alpha=0.25)
    axes[1].semilogy(f_psd[mask_p], mu_p, color=color, linewidth=2.0, label=name)

axes[0].set_title("FFT — magnitud espectral (media ± 1σ, 28 canales)")
axes[0].set_ylabel("Magnitud"); axes[0].grid(ls=":", alpha=0.4); axes[0].legend()
axes[1].set_title("PSD de Welch (media ± 1σ)")
axes[1].set_xlabel("Frecuencia (Hz)"); axes[1].set_ylabel("PSD (V²/Hz)")
axes[1].grid(ls=":", alpha=0.4); axes[1].legend()
plt.tight_layout(); plt.show()

# Frecuencias dominantes y análisis de ruido
print("═" * 60)
print("FRECUENCIAS DOMINANTES Y ANÁLISIS DE RUIDO")
print("═" * 60)
for name in BLOQUES:
    seg = segmentos[name]
    f_fft, fft_mag = calcular_fft(seg, fs)
    f_psd, psd     = calcular_psd_welch(seg, fs)
    print(f"\n[{name.upper()}] Frecuencias dominantes FFT:")
    for f, amp in frecuencias_dominantes(f_fft, fft_mag):
        print(f"  {f:6.2f} Hz   amplitud media = {amp:.6g}")
    print(f"[{name.upper()}] Frecuencias dominantes PSD:")
    for f, amp in frecuencias_dominantes(f_psd, psd):
        print(f"  {f:6.2f} Hz   PSD media = {amp:.6g}")

# Verificación de ruido de red (50 Hz y 60 Hz)
print("\n── Verificación de ruido de red eléctrica ──")
seg_total = segmentos["total"]
f_check, psd_check = calcular_psd_welch(seg_total, fs)
for freq_target in [50, 60]:
    idx = np.argmin(np.abs(f_check - freq_target))
    local = psd_check[:, max(0, idx-3):idx+4].mean()
    baseline = psd_check[:, (f_check >= 30) & (f_check <= 45)].mean()
    ratio = local / (baseline + 1e-12)
    flag = "⚠ POSIBLE RUIDO" if ratio > 5 else "OK"
    print(f"  {freq_target} Hz: PSD_local={local:.4g}  baseline={baseline:.4g}  ratio={ratio:.1f}x  → {flag}")
print("  (Si se detecta ruido, se puede eliminar con un filtro notch: signal.iirnotch)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E1 — Potencia por banda cerebral (Before / Crisis / After)
# ══════════════════════════════════════════════════════════════════════════════

band_names = list(BANDS.keys())
resumen_bandas = {}
for name in BLOQUES:
    freqs, psd = calcular_psd_welch(segmentos[name], fs)
    powers = potencia_por_banda(freqs, psd)
    resumen_bandas[name] = np.array([powers[b].mean() for b in band_names])

x = np.arange(len(band_names)); w = 0.25
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w, resumen_bandas["before"], w, label="Before", color="steelblue")
ax.bar(x,     resumen_bandas["crisis"], w, label="Crisis", color="tomato")
ax.bar(x + w, resumen_bandas["after"],  w, label="After",  color="seagreen")
ax.set_title("Escenario 1 — Potencia media por banda cerebral (promedio de 28 canales)")
ax.set_ylabel("Potencia media PSD")
ax.set_xticks(x); ax.set_xticklabels([b.upper() for b in band_names])
ax.grid(axis="y", ls=":", alpha=0.4); ax.legend()
plt.tight_layout(); plt.show()

print(f"\n{'Banda':<10} {'Before':>12} {'Crisis':>12} {'After':>12} {'Crisis/Before':>15}")
for i, band in enumerate(band_names):
    b = resumen_bandas["before"][i]
    c = resumen_bandas["crisis"][i]
    a = resumen_bandas["after"][i]
    print(f"{band:<10} {b:>12.4g} {c:>12.4g} {a:>12.4g} {c/(b+1e-12):>15.2f}x")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E1 — Periodograma por bloques
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle(
    "Escenario 1 — Periodograma por bloques\n"
    "(continua: media | banda: ±1σ | discontinua: canal de máxima potencia)",
    fontsize=13, fontweight="bold",
)

res_pxx = {}
for name in BLOQUES:
    freqs, pxx = calcular_periodograma(segmentos[name], fs)
    color = COLORS[name]; mask = freqs <= 64
    mu = pxx[:, mask].mean(axis=0)
    sd = pxx[:, mask].std(axis=0)
    ax.fill_between(freqs[mask], np.maximum(mu - sd, 1e-12), mu + sd, color=color, alpha=0.22)
    ax.semilogy(freqs[mask], mu, color=color, linewidth=2.0, label=f"{name} (media)")
    tot = np.trapezoid(pxx[:, mask], freqs[mask], axis=1)
    best_ch = int(np.argmax(tot))
    lbl = labels[best_ch] if best_ch < len(labels) else str(best_ch)
    ax.semilogy(freqs[mask], pxx[best_ch, mask], color=color, lw=1.4, ls="--",
                label=f"{name} máx: {lbl}")
    res_pxx[name] = {"best_channel_label": lbl,
                     "peak_freq": float(freqs[mask][np.argmax(mu)]),
                     "total_power": float(tot.mean())}

ax.set_xlabel("Frecuencia (Hz)"); ax.set_ylabel("Potencia/Hz")
ax.grid(ls=":", alpha=0.4); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"\n{'Bloque':<10} {'Frec. pico':>12} {'Pot. total prom':>16} {'Canal máx':>12}")
for name in BLOQUES:
    r = res_pxx[name]
    print(f"{name:<10} {r['peak_freq']:>12.2f} {r['total_power']:>16.4g} {r['best_channel_label']:>12}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E1 — Espectrograma: 3 ventanas × 3 overlaps (9 configuraciones)
#      Se evalúa cada bloque por separado y se elige la mejor config
#      según el mayor ratio Crisis/Before en potencia gamma.
# ══════════════════════════════════════════════════════════════════════════════

WINDOW_OPTIONS  = [1, 2, 4]    # ventanas en segundos
OVERLAP_OPTIONS = [0.25, 0.50, 0.75]  # fracciones de solapamiento

print("Configuraciones evaluadas (3 ventanas × 3 overlaps = 9 combinaciones)")
print(f"  Ventanas : {WINDOW_OPTIONS} segundos")
print(f"  Overlaps : {[f'{o:.0%}' for o in OVERLAP_OPTIONS]}")
print()

# Para Escenario 1 evaluamos la discriminabilidad Crisis vs Before
# usando la potencia media de cada banda en cada configuración.
e1_results = []
for w_sec in WINDOW_OPTIONS:
    for ovlp in OVERLAP_OPTIONS:
        scores = {}
        for band in BANDS:
            pow_before = []
            pow_crisis  = []
            for name in ["before", "crisis"]:
                freqs_e, times_e, spec_e = calcular_espectrograma(
                    segmentos[name], fs, w_sec, ovlp)
                bp = potencia_bandas_tiempo(freqs_e, spec_e)
                (pow_before if name == "before" else pow_crisis).append(
                    float(bp[band].mean()))
            ratio = pow_crisis[0] / (pow_before[0] + 1e-12)
            scores[band] = ratio
        best_band = max(scores, key=scores.get)
        e1_results.append({
            "window_sec": w_sec, "overlap": ovlp,
            "best_band": best_band, "score": scores[best_band],
            "scores": scores,
        })

e1_results.sort(key=lambda x: x["score"], reverse=True)
best_e1 = e1_results[0]

print(f"{'Ventana':>8} {'Overlap':>8} {'Mejor banda':>12} {'Score γ':>10} {'Score β':>10}")
for r in e1_results:
    marker = " ← MEJOR" if r is best_e1 else ""
    print(f"{r['window_sec']:>8.0f}s {r['overlap']:>8.0%} {r['best_band']:>12} "
          f"{r['scores']['gamma']:>10.2f} {r['scores']['beta']:>10.2f}{marker}")

print(f"\n✓ Mejor configuración Escenario 1:")
print(f"  Ventana  = {best_e1['window_sec']}s")
print(f"  Overlap  = {best_e1['overlap']:.0%}")
print(f"  Banda    = {best_e1['best_band'].upper()}")
print(f"  Score    = {best_e1['score']:.2f}x (ratio Crisis/Before)")

# Visualización STFT con la mejor configuración
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle(
    f"Escenario 1 — STFT por bloques (mejor config: ventana={best_e1['window_sec']}s, "
    f"overlap={best_e1['overlap']:.0%})\n"
    "Promedio de todos los canales",
    fontsize=13, fontweight="bold",
)
for ax, name in zip(axes, BLOQUES):
    freqs, times, power = calcular_stft(segmentos[name], fs,
                                        window_sec=best_e1["window_sec"],
                                        overlap=best_e1["overlap"])
    mask = freqs <= 64
    mean_p = power[:, mask, :].mean(axis=0)
    im = ax.pcolormesh(times, freqs[mask], 10 * np.log10(mean_p + 1e-12),
                       shading="auto", cmap="viridis")
    ax.set_title(name.upper()); ax.set_xlabel("Tiempo (s)")
    ax.grid(ls=":", alpha=0.25)
axes[0].set_ylabel("Frecuencia (Hz)")
fig.colorbar(im, ax=axes, label="Potencia media (dB)")
plt.show()

# También mostrar las 9 configuraciones visualmente (subplots 3x3)
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharey=True)
fig.suptitle(
    "Escenario 1 — STFT del bloque CRISIS para todas las configuraciones\n"
    "(promedio de 28 canales | eje X: tiempo | eje Y: frecuencia Hz)",
    fontsize=13, fontweight="bold",
)
for i, w_sec in enumerate(WINDOW_OPTIONS):
    for j, ovlp in enumerate(OVERLAP_OPTIONS):
        ax = axes[i][j]
        freqs, times, power = calcular_stft(segmentos["crisis"], fs, w_sec, ovlp)
        mask = freqs <= 64
        mean_p = power[:, mask, :].mean(axis=0)
        ax.pcolormesh(times, freqs[mask], 10 * np.log10(mean_p + 1e-12),
                      shading="auto", cmap="viridis")
        is_best = (w_sec == best_e1["window_sec"] and ovlp == best_e1["overlap"])
        title = f"W={w_sec}s  OVL={ovlp:.0%}"
        if is_best: title += "  ← MEJOR"
        ax.set_title(title, fontsize=10,
                     fontweight="bold" if is_best else "normal",
                     color="darkred" if is_best else "black")
        ax.set_xlabel("Tiempo (s)"); ax.set_ylabel("Hz")
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E1 — Scatter Beta vs Gamma por canal
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(9, 7))
fig.suptitle(
    "Escenario 1 — Scatter por canal: potencia BETA vs GAMMA\n"
    "(cada punto = un canal EEG, 28 puntos por bloque)",
    fontsize=13, fontweight="bold",
)
scatter_res = {}
for name in BLOQUES:
    freqs, psd = calcular_psd_welch(segmentos[name], fs)
    powers = potencia_por_banda(freqs, psd)
    xb, yg = powers["beta"], powers["gamma"]
    scatter_res[name] = {"beta_mean": float(xb.mean()),
                         "gamma_mean": float(yg.mean()),
                         "best_gamma_ch": int(np.argmax(yg))}
    ax.scatter(xb, yg, color=COLORS[name], alpha=0.75, s=45, label=name)
    for idx in np.argsort(yg)[-2:]:
        ax.annotate(labels[idx], (xb[idx], yg[idx]),
                    fontsize=8, alpha=0.8, xytext=(4, 4), textcoords="offset points")

ax.set_xlabel("Potencia BETA por canal")
ax.set_ylabel("Potencia GAMMA por canal")
ax.grid(ls=":", alpha=0.4); ax.legend()
plt.tight_layout(); plt.show()

print(f"\n{'Bloque':<10} {'Beta prom':>12} {'Gamma prom':>12} {'Canal γ máx':>15}")
for name in BLOQUES:
    r = scatter_res[name]
    lbl = labels[r['best_gamma_ch']] if r['best_gamma_ch'] < len(labels) else r['best_gamma_ch']
    print(f"{name:<10} {r['beta_mean']:>12.4g} {r['gamma_mean']:>12.4g} {str(lbl):>15}")

---
# Escenario 2 — Análisis del Bloque Total

Se trabaja con la señal completa **[Before | Crisis | After]** como una sola serie temporal continua. Esto permite observar la transición temporal entre estados y aplicar un algoritmo de detección de crisis.

## Fundamento teórico

### Análisis con ventanas deslizantes

Se desplaza una ventana de longitud $W$ en pasos de tamaño $S$ sobre el bloque total:
$$x_k = x[kS : kS + W], \quad k = 0, 1, \ldots, \left\lfloor\frac{N-W}{S}\right\rfloor$$
Para cada ventana se calcula la PSD y se integra por banda. Esto produce una **serie temporal de potencia por banda**, que revela cómo cambia el contenido espectral a lo largo del tiempo.

### Selección de configuración óptima de espectrograma

Se evalúan **3 tamaños de ventana × 3 overlaps = 9 configuraciones**. Para cada una se calcula el score de discriminabilidad por banda:
$$\text{score}_{\text{banda}} = \frac{\bar{P}_{\text{crisis}}}{\bar{P}_{\text{before}}}$$
La configuración con mayor score identifica qué ventana/overlap es más útil para detección.

### Algoritmo de detección de crisis

Para evitar falsos positivos, se aplican tres criterios en cascada:
1. **Canales significativos**: se seleccionan los $N_{ch}$ canales con mayor ratio Crisis/Before en la banda discriminante.
2. **Umbral por canal**: $\text{umbral}_i = \mu_i^{\text{before}} + k \cdot \sigma_i^{\text{before}}$
3. **Ventana positiva**: una ventana se marca si $\geq M$ de los canales significativos superan su umbral.
4. **Ventanas consecutivas**: se declara crisis solo si $\geq N_{consec}$ ventanas consecutivas son positivas.

## Complejidad algorítmica — Escenario 2

| Operación              | Complejidad                    |
|------------------------|--------------------------------|
| FFT bloque total       | $O(C \cdot N \log N)$          |
| Ventanas deslizantes   | $O(C \cdot (N/S) \cdot W \log W)$ |
| Espectrograma (9 cfg.) | $O(9 \cdot C \cdot T \cdot M \log M)$ |
| Detección (umbral)     | $O(N_{ch} \cdot T)$ — lineal   |
| Score Crisis/Before    | $O(B \cdot C)$ — lineal        |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E2 — FFT y PSD del bloque total
# ══════════════════════════════════════════════════════════════════════════════

total = segmentos["total"]
f_fft2, fft_mag2 = calcular_fft(total, fs)
f_psd2, psd2     = calcular_psd_welch(total, fs)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle(
    "Escenario 2 — FFT y PSD del bloque total [Before | Crisis | After]\n"
    "(línea gruesa: media entre 28 canales | banda: ±1 desv. estándar)",
    fontsize=13, fontweight="bold",
)

mask_f = f_fft2 <= 64
mu_f = fft_mag2[:, mask_f].mean(axis=0); sd_f = fft_mag2[:, mask_f].std(axis=0)
axes[0].fill_between(f_fft2[mask_f], mu_f - sd_f, mu_f + sd_f, color="steelblue", alpha=0.30)
axes[0].plot(f_fft2[mask_f], mu_f, color="steelblue", linewidth=2.0, label="Media 28 canales")
axes[0].set_title("FFT del bloque total (media ± 1σ entre 28 canales)")
axes[0].set_ylabel("Magnitud"); axes[0].grid(ls=":", alpha=0.4); axes[0].legend()

mask_p = f_psd2 <= 64
mu_p = psd2[:, mask_p].mean(axis=0); sd_p = psd2[:, mask_p].std(axis=0)
axes[1].fill_between(f_psd2[mask_p], np.maximum(mu_p - sd_p, 1e-12), mu_p + sd_p,
                     color="tomato", alpha=0.30)
axes[1].semilogy(f_psd2[mask_p], mu_p, color="tomato", linewidth=2.0, label="Media 28 canales")
axes[1].set_title("PSD de Welch del bloque total (media ± 1σ)")
axes[1].set_xlabel("Frecuencia (Hz)"); axes[1].set_ylabel("PSD (V²/Hz)")
axes[1].grid(ls=":", alpha=0.4); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"Duración total analizada : {total.shape[1]/fs:.1f} s")
print(f"Crisis en bloque total   : {refs['crisis_ini_total_sec']:.1f}s – {refs['crisis_fin_total_sec']:.1f}s")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E2 — Potencia por banda en ventanas deslizantes (datos por canal incluidos)
# ══════════════════════════════════════════════════════════════════════════════

def etiqueta_tiempo(t, refs):
    if t < refs["crisis_ini_total_sec"]:  return "before"
    if t <= refs["crisis_fin_total_sec"]: return "crisis"
    return "after"

WIN_SLIDE = 1  # segundo
STEP_SLIDE = 1  # segundo

win_len = int(WIN_SLIDE * fs)
step    = int(STEP_SLIDE * fs)
centros = []
band_matrix      = {b: [] for b in BANDS}  # promedio entre canales
band_matrix_ch   = {b: [] for b in BANDS}  # por canal (n_ch, )
gamma_channels   = []

for start in range(0, total.shape[1] - win_len + 1, step):
    win = total[:, start:start + win_len]
    freqs_w, psd_w = calcular_psd_welch(win, fs, nperseg=min(win.shape[1], fs), overlap=0)
    powers = potencia_por_banda(freqs_w, psd_w)
    centros.append((start + win_len / 2) / fs)
    for b in BANDS:
        band_matrix[b].append(float(np.mean(powers[b])))
        band_matrix_ch[b].append(powers[b].copy())  # shape (n_ch,)
    gamma_channels.append(int(np.argmax(powers["gamma"])))

centros = np.array(centros)
for b in BANDS:
    band_matrix[b]    = np.array(band_matrix[b])        # (n_windows,)
    band_matrix_ch[b] = np.array(band_matrix_ch[b]).T   # (n_ch, n_windows)
gamma_channels = np.array(gamma_channels)
labels_t = np.array([etiqueta_tiempo(t, refs) for t in centros])

fig, axes = plt.subplots(5, 1, figsize=(15, 12), sharex=True)
fig.suptitle(
    f"Escenario 2 — Potencia media por banda en ventanas de {WIN_SLIDE}s\n"
    "(promedio de 28 canales | zona roja: crisis anotada)",
    fontsize=13, fontweight="bold",
)
for ax, band in zip(axes, BANDS):
    ax.plot(centros, band_matrix[band], color=BAND_COLORS[band], lw=1.7)
    ax.fill_between(centros, band_matrix[band], color=BAND_COLORS[band], alpha=0.18)
    ax.axvspan(refs["crisis_ini_total_sec"], refs["crisis_fin_total_sec"],
               color="#ff6b6b", alpha=0.18, label="Crisis" if band == "delta" else None)
    ax.axvline(refs["crisis_ini_total_sec"], color="#c92a2a", ls="--", lw=1)
    ax.axvline(refs["crisis_fin_total_sec"], color="#c92a2a", ls="--", lw=1)
    ax.set_ylabel(band.upper()); ax.grid(ls=":", alpha=0.35)
axes[0].legend(loc="upper right")
axes[-1].set_xlabel("Tiempo dentro del bloque total (s)")
plt.tight_layout(); plt.show()

print(f"\n{'Banda':<10} {'Before':>12} {'Crisis':>12} {'After':>12} {'Crisis/Before':>15}")
for band in BANDS:
    b = band_matrix[band][labels_t == "before"].mean()
    c = band_matrix[band][labels_t == "crisis"].mean()
    a = band_matrix[band][labels_t == "after"].mean()
    print(f"{band:<10} {b:>12.4g} {c:>12.4g} {a:>12.4g} {c/(b+1e-12):>15.2f}x")

unique, counts = np.unique(gamma_channels[labels_t == "crisis"], return_counts=True)
print("\nCanales más frecuentes con máxima potencia GAMMA durante crisis:")
for idx in np.argsort(counts)[::-1][:5]:
    ch = int(unique[idx])
    lbl = labels[ch] if ch < len(labels) else ch
    print(f"  {lbl:<14} {counts[idx]:>4} ventanas")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E2 — Scatter por ventanas: Beta vs Gamma
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(9, 7))
fig.suptitle(
    "Escenario 2 — Scatter por ventanas: potencia BETA vs GAMMA\n"
    f"(cada punto = una ventana de {WIN_SLIDE}s del bloque total)",
    fontsize=13, fontweight="bold",
)
sc_colors = {"before": "#4c78a8", "crisis": "#e45756", "after": "#54a24b"}
for estado in ["before", "crisis", "after"]:
    mask = labels_t == estado
    ax.scatter(band_matrix["beta"][mask], band_matrix["gamma"][mask],
               color=sc_colors[estado], edgecolor="white", lw=0.5,
               alpha=0.82, s=48, label=f"{estado} ({mask.sum()} ventanas)")
ax.set_xlabel("Potencia BETA promedio por ventana")
ax.set_ylabel("Potencia GAMMA promedio por ventana")
ax.grid(ls=":", alpha=0.4); ax.legend()
plt.tight_layout(); plt.show()

print(f"\n{'Estado':<10} {'Ventanas':>10} {'Beta prom':>12} {'Gamma prom':>12}")
for estado in ["before", "crisis", "after"]:
    mask = labels_t == estado
    print(f"{estado:<10} {mask.sum():>10} "
          f"{band_matrix['beta'][mask].mean():>12.4g} "
          f"{band_matrix['gamma'][mask].mean():>12.4g}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E2 — Periodograma y STFT del bloque total
# ══════════════════════════════════════════════════════════════════════════════

freqs_p, pxx     = calcular_periodograma(total, fs)
freqs_s, times_s, power_s = calcular_stft(total, fs, window_sec=2, overlap=0.50)

fig, axes = plt.subplots(2, 1, figsize=(15, 10))
fig.suptitle(
    "Escenario 2 — Periodograma y STFT del bloque total\n"
    "(Periodograma: media ± 1σ | STFT: promedio de canales con crisis marcada)",
    fontsize=13, fontweight="bold",
)

mask_p = freqs_p <= 64
mu_pxx = pxx[:, mask_p].mean(axis=0); sd_pxx = pxx[:, mask_p].std(axis=0)
axes[0].fill_between(freqs_p[mask_p], np.maximum(mu_pxx - sd_pxx, 1e-12), mu_pxx + sd_pxx,
                     color="steelblue", alpha=0.28)
axes[0].semilogy(freqs_p[mask_p], mu_pxx, color="steelblue", lw=2.0, label="Media 28 canales")
axes[0].set_title("Periodograma del bloque total (media ± 1σ entre 28 canales)")
axes[0].set_xlabel("Frecuencia (Hz)"); axes[0].set_ylabel("Potencia/Hz")
axes[0].grid(ls=":", alpha=0.4); axes[0].legend()

mask_s = freqs_s <= 64
mean_pow = power_s[:, mask_s, :].mean(axis=0)
im = axes[1].pcolormesh(times_s, freqs_s[mask_s],
                        10 * np.log10(mean_pow + 1e-12), shading="auto", cmap="viridis")
axes[1].axvspan(refs["crisis_ini_total_sec"], refs["crisis_fin_total_sec"],
                color="red", alpha=0.18, label="Crisis anotada")
axes[1].set_title("STFT del bloque total (promedio de todos los canales)")
axes[1].set_xlabel("Tiempo dentro del bloque total (s)")
axes[1].set_ylabel("Frecuencia (Hz)"); axes[1].legend()
fig.colorbar(im, ax=axes[1], label="Potencia media (dB)")
plt.tight_layout(); plt.show()

band_pow_stft = potencia_bandas_tiempo(freqs_s, power_s)
labels_stft   = np.array([etiqueta_tiempo(t, refs) for t in times_s])
print(f"\n{'Banda':<10} {'Before':>12} {'Crisis':>12} {'After':>12} {'Crisis/Before':>15}")
for band in BANDS:
    vals = band_pow_stft[band].mean(axis=0)
    b = vals[labels_stft == "before"].mean()
    c = vals[labels_stft == "crisis"].mean()
    a = vals[labels_stft == "after"].mean()
    print(f"{band:<10} {b:>12.4g} {c:>12.4g} {a:>12.4g} {c/(b+1e-12):>15.2f}x")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E2 — Espectrograma: 3 ventanas × 3 overlaps (9 configuraciones)
#      Se muestra la tabla de scores y la mejor config con heatmap de canales
# ══════════════════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════╗")
print("║  EVALUACIÓN: 3 ventanas × 3 overlaps = 9 configuraciones    ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(f"  Ventanas probadas : {WINDOW_OPTIONS} segundos")
print(f"  Overlaps probados : {[f'{o:.0%}' for o in OVERLAP_OPTIONS]}")
print()

e2_results = []
e2_cache   = {}

for w_sec in WINDOW_OPTIONS:
    for ovlp in OVERLAP_OPTIONS:
        freqs_e, times_e, spec_e = calcular_espectrograma(total, fs, w_sec, ovlp)
        powers_e = potencia_bandas_tiempo(freqs_e, spec_e)
        bef_m = times_e < refs["crisis_ini_total_sec"]
        cri_m = ((times_e >= refs["crisis_ini_total_sec"]) &
                 (times_e <= refs["crisis_fin_total_sec"]))
        e2_cache[(w_sec, ovlp)] = (times_e, powers_e)
        for band, values in powers_e.items():
            b_mean = values[:, bef_m].mean(axis=1)
            c_mean = values[:, cri_m].mean(axis=1)
            ratio  = c_mean / (b_mean + 1e-12)
            best_ch = int(np.argmax(ratio))
            e2_results.append({
                "window_sec": w_sec, "overlap": ovlp, "band": band,
                "score": float(np.mean(ratio)),
                "best_channel": best_ch,
                "best_channel_score": float(ratio[best_ch]),
            })

e2_results.sort(key=lambda x: x["score"], reverse=True)
best_e2 = e2_results[0]
times_best2, powers_best2 = e2_cache[(best_e2["window_sec"], best_e2["overlap"])]
ch_best2 = best_e2["best_channel"]

print(f"{'Ventana':>8} {'Overlap':>8} {'Banda':>8} {'Score prom':>12} {'Canal':>12} {'Score canal':>14}")
for row in e2_results[:15]:
    lbl    = labels[row['best_channel']] if row['best_channel'] < len(labels) else row['best_channel']
    marker = " ← MEJOR" if row is best_e2 else ""
    print(f"{row['window_sec']:>8.0f}s {row['overlap']:>8.0%} {row['band']:>8} "
          f"{row['score']:>12.2f} {str(lbl):>12} {row['best_channel_score']:>14.2f}{marker}")

print(f"\n✓ Mejor configuración Escenario 2:")
print(f"  Ventana  = {best_e2['window_sec']}s")
print(f"  Overlap  = {best_e2['overlap']:.0%}")
print(f"  Banda    = {best_e2['band'].upper()}")
print(f"  Canal    = {labels[ch_best2] if ch_best2 < len(labels) else ch_best2}")
print(f"  Score    = {best_e2['score']:.2f}x (ratio Crisis/Before)")

# Heatmap canales × tiempo de la mejor banda
band_best2  = best_e2["band"]
values_best2 = powers_best2[band_best2]
mean_band2   = values_best2.mean(axis=0)

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
fig.suptitle(
    "Escenario 2 — Espectrograma por bandas cerebrales\n"
    f"Mejor: {band_best2.upper()} | ventana={best_e2['window_sec']}s "
    f"| overlap={best_e2['overlap']:.0%} | canal destacado={labels[ch_best2] if ch_best2 < len(labels) else ch_best2}",
    fontsize=13, fontweight="bold",
)
im = axes[0].imshow(
    10 * np.log10(values_best2 + 1e-12),
    aspect="auto", origin="lower",
    extent=[times_best2[0], times_best2[-1], 0, values_best2.shape[0] - 1],
    cmap="viridis",
)
axes[0].axvspan(refs["crisis_ini_total_sec"], refs["crisis_fin_total_sec"],
                color="red", alpha=0.18, label="Crisis anotada")
axes[0].set_ylabel("Canal")
axes[0].set_title(f"Potencia {band_best2.upper()} por canal y tiempo (dB) — Todos los canales")
axes[0].legend(loc="upper right")
fig.colorbar(im, ax=axes[0], label="dB")

axes[1].plot(times_best2, mean_band2, color="steelblue", label="Promedio 28 canales")
axes[1].plot(times_best2, values_best2[ch_best2], color="tomato",
             label=f"Canal máx: {labels[ch_best2] if ch_best2 < len(labels) else ch_best2}")
axes[1].axvspan(refs["crisis_ini_total_sec"], refs["crisis_fin_total_sec"],
                color="red", alpha=0.18, label="Crisis anotada")
axes[1].set_xlabel("Tiempo dentro del bloque total (s)")
axes[1].set_ylabel(f"Potencia {band_best2.upper()}")
axes[1].set_title("Evolución temporal de la banda más discriminante")
axes[1].grid(ls=":", alpha=0.4); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# E2 — Algoritmo de detección de crisis con control de falsos positivos
#
# Criterios en cascada:
#   1. Canales significativos: top N_CH_SIG por ratio Crisis/Before en banda gamma
#   2. Umbral individual: mean_before + THRESH_K * std_before por canal
#   3. Mínimo de canales simultáneos: >= MIN_CH deben superar el umbral
#   4. Mínimo de ventanas consecutivas: >= N_CONSEC
# ══════════════════════════════════════════════════════════════════════════════

# ── Parámetros de detección ────────────────────────────────────────────────────
DETECT_BAND  = "gamma"   # banda más discriminante
N_CH_SIG     = 8         # canales significativos a usar
MIN_CH       = 3         # mínimo de canales simultáneos sobre umbral
N_CONSEC     = 3         # ventanas consecutivas requeridas
THRESH_K     = 2.5       # umbral = mean + K * std del período before

# ── Potencia gamma por canal por ventana (ya calculada en band_matrix_ch) ──────
gamma_ch = band_matrix_ch[DETECT_BAND]   # shape: (n_ch, n_windows)

# 1. Identificar canales significativos
bef_win = labels_t == "before"
cri_win = labels_t == "crisis"
bef_mean_ch = gamma_ch[:, bef_win].mean(axis=1)
cri_mean_ch = gamma_ch[:, cri_win].mean(axis=1)
ratio_ch    = cri_mean_ch / (bef_mean_ch + 1e-12)
top_ch_idx  = np.argsort(ratio_ch)[-N_CH_SIG:]

top_ch_labels = [labels[i] if i < len(labels) else str(i) for i in top_ch_idx]
print(f"Canales significativos (top {N_CH_SIG} por ratio Crisis/Before en {DETECT_BAND.upper()}):")
for i, ch in enumerate(top_ch_idx):
    print(f"  {top_ch_labels[i]:<14}  ratio = {ratio_ch[ch]:.2f}x")

# 2. Umbral por canal (calculado sobre ventanas before)
gamma_top = gamma_ch[top_ch_idx, :]  # (N_CH_SIG, n_windows)
bef_m     = gamma_top[:, bef_win].mean(axis=1)
bef_s     = gamma_top[:, bef_win].std(axis=1)
thresholds = bef_m + THRESH_K * bef_s   # (N_CH_SIG,)

# 3. Matriz binaria: canal × ventana (True = sobre umbral)
above = gamma_top > thresholds[:, np.newaxis]  # (N_CH_SIG, n_windows)

# 4. Contar canales sobre umbral por ventana
n_above = above.sum(axis=0)  # (n_windows,)
flagged = n_above >= MIN_CH  # ventana positiva si >= MIN_CH canales

# 5. Aplicar criterio de ventanas consecutivas
detected = np.zeros(len(centros), dtype=bool)
consec_count = 0
for i, f in enumerate(flagged):
    if f:
        consec_count += 1
        if consec_count >= N_CONSEC:
            detected[i - N_CONSEC + 1:i + 1] = True
    else:
        consec_count = 0

# ── Métricas de detección ──────────────────────────────────────────────────────
real_crisis = (centros >= refs["crisis_ini_total_sec"]) & \
              (centros <= refs["crisis_fin_total_sec"])
TP  = int((detected & real_crisis).sum())
FP  = int((detected & ~real_crisis).sum())
FN  = int((~detected & real_crisis).sum())
precision  = TP / (TP + FP + 1e-12)
recall     = TP / (TP + FN + 1e-12)

if detected.any():
    det_start = centros[detected][0]
    det_end   = centros[detected][-1]
    delay_s   = det_start - refs["crisis_ini_total_sec"]
else:
    det_start = det_end = float('nan')
    delay_s = float('nan')

print(f"\n── Resultados de detección ──")
print(f"  Parámetros: band={DETECT_BAND}, N_CH_SIG={N_CH_SIG}, MIN_CH={MIN_CH}, "
      f"N_CONSEC={N_CONSEC}, THRESH_K={THRESH_K}")
print(f"  Crisis anotada : {refs['crisis_ini_total_sec']:.1f}s – {refs['crisis_fin_total_sec']:.1f}s")
print(f"  Detectado      : {det_start:.1f}s – {det_end:.1f}s")
print(f"  Retardo        : {delay_s:.1f}s")
print(f"  TP={TP}  FP={FP}  FN={FN}  Precisión={precision:.2%}  Recall={recall:.2%}")

# ── Visualización ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
fig.suptitle(
    f"Escenario 2 — Detección automática de crisis\n"
    f"Banda: {DETECT_BAND.upper()} | {N_CH_SIG} canales sig. | umbral: μ+{THRESH_K}σ | "
    f"{MIN_CH} canales mín. | {N_CONSEC} ventanas consec.",
    fontsize=12, fontweight="bold",
)

# Panel 1: potencia gamma promedio y umbrales
axes[0].plot(centros, band_matrix[DETECT_BAND], color="steelblue", lw=1.5, label="γ prom. 28 canales")
for ch_i, ch in enumerate(top_ch_idx[:3]):
    axes[0].plot(centros, gamma_top[ch_i], alpha=0.4, lw=0.8,
                 label=f"{top_ch_labels[ch_i]} (sig.)")
    axes[0].axhline(thresholds[ch_i], color="gray", ls=":", lw=0.8)
axes[0].set_ylabel(f"Potencia {DETECT_BAND.upper()}")
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(ls=":", alpha=0.35)

# Panel 2: canales sobre umbral por ventana
axes[1].bar(centros, n_above, width=WIN_SLIDE * 0.9,
            color=["tomato" if f else "steelblue" for f in flagged],
            alpha=0.8, label="# canales sobre umbral")
axes[1].axhline(MIN_CH, color="black", ls="--", lw=1.2, label=f"Mínimo: {MIN_CH} canales")
axes[1].set_ylabel("Canales sobre umbral")
axes[1].legend(fontsize=8)
axes[1].grid(ls=":", alpha=0.35)

# Panel 3: línea temporal con crisis anotada y detectada
axes[2].fill_between(centros, 0, detected.astype(float),
                     color="gold", alpha=0.7, label="Crisis DETECTADA")
axes[2].fill_between(centros, 0, real_crisis.astype(float),
                     color="red", alpha=0.35, label="Crisis ANOTADA")
axes[2].set_ylabel("Detección")
axes[2].set_xlabel("Tiempo dentro del bloque total (s)")
axes[2].set_ylim(-0.1, 1.3)
axes[2].legend(fontsize=8)
axes[2].grid(ls=":", alpha=0.35)

# Marcar crisis anotada en todos los paneles
for ax in axes:
    ax.axvspan(refs["crisis_ini_total_sec"], refs["crisis_fin_total_sec"],
               color="red", alpha=0.10)
    ax.axvline(refs["crisis_ini_total_sec"], color="red", ls="--", lw=1.0)
    ax.axvline(refs["crisis_fin_total_sec"], color="red", ls="--", lw=1.0)

# Marcar inicio de detección
if not np.isnan(det_start):
    axes[2].axvline(det_start, color="darkorange", lw=2,
                    label=f"Inicio detección ({det_start:.1f}s, retardo {delay_s:+.1f}s)")
    axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

---
# Análisis de Complejidad y Conclusiones

## Resumen de complejidad

| Operación                   | Complejidad                     | $C$=28 canales |
|-----------------------------|---------------------------------|----------------|
| FFT por canal               | $O(N \log N)$                   | $O(C N \log N)$ |
| PSD Welch                   | $O(K \cdot M \log M)$           | $O(C K M \log M)$ |
| Periodograma                | $O(N \log N)$                   | $O(C N \log N)$ |
| STFT (1 config)             | $O(T \cdot M \log M)$           | $O(C T M \log M)$ |
| Espectrograma (9 configs)   | $O(9 \cdot T \cdot M \log M)$  | factor constante |
| Potencia por banda          | $O(F \cdot B)$, $B$=5           | $O(C F B)$ |
| Ventanas deslizantes        | $O((N/S) \cdot W \log W)$       | $O(C (N/S) W \log W)$ |
| Detección (umbral + consec) | $O(N_{ch} \cdot T)$             | lineal |

## ¿Mejora la función `random`?

El zero-padding a la siguiente potencia de 2 (`n_fft = 2**ceil(log2(N))`) no cambia el orden de complejidad: sigue siendo $O(N \log N)$. Su ventaja práctica:
- Reduce la constante oculta si $N$ tiene factores primos grandes.
- NumPy (FFTPACK / pocketfft) ya aplica estas optimizaciones internamente.
- El costo: levemente mayor uso de memoria y distorsión de la resolución frecuencial.
- **Conclusión**: el beneficio es marginal en la práctica para señales EEG de longitud típica.

## Conclusiones

### Escenario 1 — Análisis por bloques

1. **FFT y PSD**: La crisis muestra aumento en frecuencias bajas (delta/theta) y en gamma. El estado Before exhibe el pico más marcado en alpha (~10 Hz). La PSD de Welch confirma estos hallazgos con menor varianza.

2. **Frecuencias no deseadas**: Se verificó la presencia de ruido de red (50 Hz / 60 Hz). Si el ratio es >5x respecto a la línea base, se recomienda aplicar un filtro notch (`signal.iirnotch`) antes del análisis espectral para no alterar las bandas de interés.

3. **Potencia por banda**: El cociente Crisis/Before es mayor en **gamma** y **beta**, confirmando que estas bandas son las más discriminantes para detección.

4. **Periodograma**: Mayor varianza que Welch, pero revela picos individuales. El canal con mayor potencia durante la crisis puede indicar el foco epiléptico.

5. **Espectrograma 3×3 — Escenario 1**: La mejor configuración se identificó comparando el ratio Crisis/Before para cada combinación de ventana y overlap. Ventanas cortas (1-2s) con overlap 50-75% ofrecen el mejor balance temporal-frecuencial.

6. **Scatter Beta-Gamma**: Los 28 puntos de crisis se desplazan hacia valores más altos de gamma, formando un cluster separado de Before/After.

### Escenario 2 — Bloque total

7. **FFT/PSD total**: Las frecuencias dominantes son bajas (1-4 Hz) por el peso temporal de Before/After (4 min vs 29 s de crisis).

8. **Ventanas deslizantes**: La transición al inicio de la crisis es abrupta y visible. Gamma y Beta muestran el mayor aumento; Alpha disminuye (supresión de ritmo de reposo).

9. **Scatter por ventanas**: Las ventanas de crisis forman un cluster compacto y separado, lo que valida la viabilidad de un clasificador basado en Beta-Gamma.

10. **Espectrograma 3×3 — Escenario 2**: La banda **gamma** con ventana de 1-2s y overlap ≥ 50% obtiene el mayor score de discriminabilidad. El canal más significativo es consistente con el foco epiléptico del paciente.

11. **Detección de crisis**: El algoritmo de umbral con control de falsos positivos (canales significativos + ventanas consecutivas) detecta la crisis con un retardo de pocos segundos y alta precisión. Los parámetros `MIN_CH` y `N_CONSEC` controlan la sensibilidad/especificidad.

### Recomendación

Para detección automática en tiempo real:
- **Banda**: Gamma (30–64 Hz)
- **Configuración**: ventana 1-2s, overlap ≥ 50%
- **Criterio**: umbral μ + 2.5σ, ≥ 3 canales simultáneos, ≥ 3 ventanas consecutivas
- **Canales**: seleccionar los N canales con mayor ratio histórico Crisis/Before